In [12]:
from migration import ddl_resolver
from migration.ddl_resolver import DdlResolver
from src.utils.file_utils import parse_file_name
from migration.cur.decomposer import CurSqlDecomposer, CurDecomposerWriter
from migration.cur.metadata import CurMetadataProcessor
from migration.cur.generator import CurPySparkGenerator
from src.paths import *

In [13]:
USERNAME

'ext_giadung'

In [14]:
from src.utils.source_rule_loader import load_all_source_rules

# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_contact.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_broker.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_address.sql"
input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_crs.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_einvoice_customer_address.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_risk_profile.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
# input_file = DATALAKE_SCRIPT_DIR /"dml" / "cur" /  "cur_dim_customer_employment.sql"
file_name = os.path.basename(input_file).replace('.sql', '')
layer, sub_layer, source_name, base_table = parse_file_name(input_file)
output_root = PROJECT_ROOT / "output" / "migration"

all_source_rules = load_all_source_rules()
source_rules = load_all_source_rules()["cur"]

print(f"File gốc tại: {input_file}")

File gốc tại: C:\Users\ext_giadung\projects\datalake-script\dml\cur\cur_dim_crs.sql


In [23]:
# def run_migration_pipeline():
# ==========================================
# BƯỚC 1: BÓC TÁCH SQL (DECOMPOSER)
# ==========================================
decomposer = CurSqlDecomposer(source_rules)
decomposed_script = decomposer.decompose(input_file)

# Ghi file sub-SQL ra ổ đĩa
writer = CurDecomposerWriter()
writer.write(decomposed_script, output_root / file_name)
print(f"✅ [Bước 1] Đã bóc tách thành các block tại: {output_root / file_name / 'processing_steps'}")

✅ [Bước 1] Đã bóc tách thành các block tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_crs\processing_steps


In [16]:
decomposed_script.temp_table_registry

defaultdict(set,
            {'MHBOS': {'TEMP_DIM_CRS'},
             'TOMS': {'TEMP_DIM_CRS'},
             'KDI': {'TEMP_DIM_CRS'},
             'SMF': {'TEMP_DIM_CRS'},
             'SBL': {'TEMP_DIM_CRS'},
             'LMS': {'TEMP_DIM_CRS'},
             'RAK': {'TEMP_DIM_CRS'},
             'GUAVA': {'TEMP_DIM_CRS'},
             'SUNGL': {'TEMP_DIM_CRS'},
             'M21': {'TEMP_DIM_CRS'}})

In [24]:

# ==========================================
# BƯỚC 2: XỬ LÝ METADATA (PROCESSOR)
# ==========================================
processor = CurMetadataProcessor(source_rules)

try:
    # Hàm này sẽ phân tích AST, tự động tìm file DDL và trích xuất Schema
    pipeline_config = processor.process(decomposed_script, input_file)

    # Ghi file YAML
    metadata_output_dir = output_root / file_name / "metadata"
    processor.write_yaml(pipeline_config, metadata_output_dir)

    print(f"✅ [Bước 2] Đã xử lý Metadata thành công!")
    print(f"   -> Model nhận diện được: Model {pipeline_config['model_type']}")
    print(f"   -> Khóa (Key) nhận diện được: {pipeline_config['primary_key']['logical_primary_key']}")
    print(f"   -> File YAML đã lưu tại: {metadata_output_dir / (file_name + '.yaml')}")

except FileNotFoundError as e:
    print(f"❌ [Lỗi Bước 2]: {e}")
    print("💡 Gợi ý: Hãy đảm bảo bạn có file DDL tương ứng tại `docs/datalake_old/ddl/com_r_k2_cif_alias.sql` hoặc cùng thư mục `dml/` để hàm trích xuất Schema hoạt động!")


✅ [Bước 2] Đã xử lý Metadata thành công!
   -> Model nhận diện được: Model UNKNOWN
   -> Khóa (Key) nhận diện được: ['SOURCE_NAME', 'CUSTOMER_ID', 'TAXPAYER_IDENTIFICATION_NO_1']
   -> File YAML đã lưu tại: C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_crs\metadata\cur_dim_crs.yaml


In [27]:
print("==========================================")
print(" BƯỚC 3: SINH CODE PYSPARK (GENERATOR)")
print("==========================================")

with open(output_root / file_name / "metadata" / (file_name + '.yaml'), 'r') as f:
    pipeline_config = yaml.safe_load(f)

generator = CurPySparkGenerator(source_rules, output_mode="simple")
ddl_context, dml_context = generator.generate(pipeline_config, output_root / file_name)

print("🎉 Hoàn tất toàn bộ Pipeline!")

 BƯỚC 3: SINH CODE PYSPARK (GENERATOR)
Processing source: UNKNOWN_SOURCE, from file: 88_unknown_blocks
Processing source: SMF, from file: 10_source_smf
Processing source: TOMS, from file: 10_source_toms
Processing source: MHBOS, from file: 10_source_mhbos
Processing source: M21, from file: 10_source_m21
Processing source: KDI, from file: 10_source_kdi
Processing source: LMS, from file: 10_source_lms
Processing source: SBL, from file: 10_source_sbl
Processing source: SUNGL, from file: 10_source_sungl
Processing source: GUAVA, from file: 10_source_guava
Processing source: RAK, from file: 10_source_rak
🎉 Hoàn tất toàn bộ Pipeline!


In [19]:
dml_context["LMS"]

{'pipeline_id': 'cur_dim_crs',
 'generated_at': '2026-05-26 07:22:16',
 'target_table_name': 'dim_crs_lms',
 'source_name': 'LMS',
 'base_table': 'dim_crs',
 'original_columns': [{'name': 'customer_id',
   'type': 'VARCHAR(20)',
   'remark': None},
  {'name': 'crs_entity_type', 'type': 'VARCHAR(2)', 'remark': None},
  {'name': 'name_of_controlling_person_1',
   'type': 'VARCHAR(100)',
   'remark': None},
  {'name': 'name_of_controlling_person_2',
   'type': 'VARCHAR(100)',
   'remark': None},
  {'name': 'crs_country', 'type': 'VARCHAR(2)', 'remark': None},
  {'name': 'crs_tax_residence', 'type': 'VARCHAR(2)', 'remark': None},
  {'name': 'taxpayer_identification_no_1',
   'type': 'VARCHAR(20)',
   'remark': None},
  {'name': 'taxpayer_identification_no_2',
   'type': 'VARCHAR(20)',
   'remark': None},
  {'name': 'tin_unavailable_reason', 'type': 'VARCHAR(2)', 'remark': None},
  {'name': 'tin_remarks', 'type': 'VARCHAR(200)', 'remark': None},
  {'name': 'source_record_id', 'type': 'VARCH

In [20]:

from core.parser import HiveScriptParser
step_file = Path(r"C:\Users\ext_giadung\projects\hql_spark_bridge\output\migration\cur_dim_address\processing_steps\10_source_mhbos.sql")
context = HiveScriptParser.parse_file(str(step_file))
context.source_name = "MHBOS"


jinja_render_model = generator.transformer.transform(pipeline_config, context)
jinja_render_model.transformed_queries

[{'type': 'query',
  'content': '/* ==============[Group.1]============== */\nDROP TABLE IF EXISTS {params["cur_schema"]}.TEMP_DIM_ACCOUNT_ADDRESS'},
 {'type': 'query',
  'content': 'CREATE TABLE {params["cur_schema"]}.TEMP_DIM_ACCOUNT_ADDRESS (\n  OWNER_ID VARCHAR(50), /* None */\n  ADDRESS_OWNER_TYPE VARCHAR(20), /* None */\n  ADDRESS_TYPE VARCHAR(10), /* None */\n  ADDRESS_LINE_1 VARCHAR(100), /* None */\n  ADDRESS_LINE_2 VARCHAR(100), /* None */\n  ADDRESS_LINE_3 VARCHAR(100), /* None */\n  ADDRESS_LINE_4 VARCHAR(100), /* None */\n  CITY VARCHAR(255), /* None */\n  STATE VARCHAR(50), /* None */\n  POSTCODE VARCHAR(5), /* None */\n  COUNTRY VARCHAR(3), /* None */\n  ADDRESS_CREATE_DATE DATE, /* None */\n  ADDRESS_UPDATE_DATE DATE, /* None */\n  LINE_OF_BUSINESS VARCHAR(20), /* None */\n  SOURCE_NAME VARCHAR(10), /* None */\n  SOURCE_RECORD_ID VARCHAR(50) /* None */\n)\nUSING PARQUET\nTBLPROPERTIES (\n  \'parquet.compression\'=\'SNAPPY\',\n  \'external.table.purge\'=\'true\'\n)'},
 {

In [21]:
ddl_resolver  = DdlResolver()

ddl_resolver.enrich(pipeline_config, model_type="1")

{'columns': [{'name': 'customer_id', 'type': 'VARCHAR(20)', 'remark': None},
  {'name': 'crs_entity_type', 'type': 'VARCHAR(2)', 'remark': None},
  {'name': 'name_of_controlling_person_1',
   'type': 'VARCHAR(100)',
   'remark': None},
  {'name': 'name_of_controlling_person_2',
   'type': 'VARCHAR(100)',
   'remark': None},
  {'name': 'crs_country', 'type': 'VARCHAR(2)', 'remark': None},
  {'name': 'crs_tax_residence', 'type': 'VARCHAR(2)', 'remark': None},
  {'name': 'taxpayer_identification_no_1',
   'type': 'VARCHAR(20)',
   'remark': None},
  {'name': 'taxpayer_identification_no_2',
   'type': 'VARCHAR(20)',
   'remark': None},
  {'name': 'tin_unavailable_reason', 'type': 'VARCHAR(2)', 'remark': None},
  {'name': 'tin_remarks', 'type': 'VARCHAR(200)', 'remark': None},
  {'name': 'source_record_id', 'type': 'VARCHAR(20)', 'remark': None},
  {'name': 'priority_level', 'type': 'INT', 'remark': None},
  {'name': 'source_update_date', 'type': 'TIMESTAMP', 'remark': None}],
 'extra_field

In [22]:
template_dir = PROJECT_ROOT / "template" / "migration"
model_folders = [d for d in template_dir.iterdir() if d.is_dir() and d.name.startswith('model_')]

for model_folder in model_folders:
    model_name = model_folder.name
    print(model_name.split("_")[-1].lower())
    # print(f"Processing model: {model_name}")
    # enricher = DdlResolver(source_rules=source_rules)
    # ddl_context = enricher.enrich(pipeline_config, model_type=model_name.split()[-1].lower())

1
2a
2b
3
3a
3b
4
5a
5b
6
